# Import World Bank API data
- Author: Bryan Bravo
- Created: 2026-03-23
## Import Libraries

In [4]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)


# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [5]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

### Variables

In [11]:
end_date = proj_vars.end_date
end_date[:7]

'2026-03'

## Query

In [7]:
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}
countries = [country for country in country_mapping.values()]


In [24]:
(f"https://data360api.worldbank.org/data360/data?DATABASE_ID=IMF_IFS&INDICATOR=IMF_IFS_RAXG&REF_AREA={', '.join(countries)}" +
    f"&FREQ=M&timePeriodFrom=2006-01&timePeriodTo={end_date[:7]}&skip=0")

'https://data360api.worldbank.org/data360/data?DATABASE_ID=IMF_IFS&INDICATOR=IMF_IFS_RAXG&REF_AREA=AUS, BRA, CAN, CHN, EU, FRA, DEU, IND, ITA, JPN, MEX, KOR, RUS, ZAF, TUR, GBR, USA&FREQ=M&timePeriodFrom=2006-01&timePeriodTo=2026-03&skip=0'

### Import FX monthly reserves

In [ ]:
# response = requests.get(
#     f"https://data360api.worldbank.org/data360/data?DATABASE_ID=IMF_IFS&INDICATOR=IMF_IFS_RAXG&REF_AREA={'AUS'}" +
#     f"&FREQ=M&timePeriodFrom=2006-01&timePeriodTo={end_date[:7]}&skip=0"
# )
# wb_data = response.json()

# if 'error_code' in wb_data:  # check if response is None
#     print(f"✗ WB API error: {wb_data.get('error_code', 'error_message')}")

# wb_df = pd.DataFrame(wb_data['value'])
# wb_df.columns = [col.lower() for col in wb_df.columns]
# wb_df = wb_df[['obs_value', 'ref_area', 'time_period']]

In [34]:
def import_wb_data(country_code, end_date):
    response = requests.get(
        f"https://data360api.worldbank.org/data360/data?DATABASE_ID=IMF_IFS&INDICATOR=IMF_IFS_RAXG&REF_AREA={country_code}" +
        f"&FREQ=M&timePeriodFrom=2006-01&timePeriodTo={end_date[:7]}&skip=0"
    )
    wb_data = response.json()

    if 'error_code' in wb_data:  # check if response is None
        print(f"✗ WB API error: {wb_data.get('error_code', 'error_message')}")

    wb_df = pd.DataFrame(wb_data['value'])
    wb_df.columns = [col.lower() for col in wb_df.columns]
    wb_df = wb_df[['obs_value', 'ref_area', 'time_period']]
    return wb_df

In [ ]:
wb_df = pd.DataFrame(columns=['obs_value', 'ref_area', 'time_period'])

wb_df 

In [38]:
wb_df

,obs_value,ref_area,time_period
